# Sentinel-2 Imagery Extraction — Rare Earth Sites, Kachin State
Goal: Extracts cloud-free, atmospherically corrected Sentinel-2 imagery clipped to bounding box which is based on polygons drawn in Google Earth (KML).

Steps:
- Initialization of Google Earth Engine
- Loading Shape file
- Creating Bounding Box
- Setting up Data Pre-processing pipeline before extracting satellite imagery from Sentinel-2 using API
- Filtering and processing Sentinel-2 collection
- Exporting to Google Drive
- Checking the collection status (Completed or Stopped)

In [87]:
import ee
import geopandas as gpd

In [98]:
# --- Initialize Google Earth Engine ---
try:
    Project_ID = 'rareearth-sites-kachin-state'
    ee.Initialize(project=Project_ID)
    print("Google Earth Engine initialized successfully.")
except Exception as e:
    print("Initialization failed. Ensure you ran 'earthengine authenticate' in your terminal.")
    print(e)

Google Earth Engine initialized successfully.


In [ ]:
# --- Load shape polygon drawn in Google Earth ---
shp_path = r"../region_of_interest/roi_polygon.shp"
gdf = gpd.read_file(shp_path)

# Ensuring CRS is WGS84 (required by GEE)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

#Check the results
print(f"Loaded {len(gdf)} polygon(s) from shape file")  #find the number of polygons in shape file
print(f"CRS: {gdf.crs}")                                #CRS
print(f"Bounds: {gdf.total_bounds}")                    #Bounding Box data

gdf

Loaded 3 feature(s) from shape file
CRS: EPSG:4326
Bounds: [98.22726679 25.57770732 98.45338589 25.7374516 ]


,id,Name,descriptio,timestamp,begin,end,altitudeMo,tessellate,extrude,visibility,drawOrder,icon,geometry
0,018752F9DA3BFF4F4E11,Region of Interest (ROI_whole),None,None,None,None,None,-1,0,0,NaN,None,"POLYGON ((98.43932 25.69777, 98.41114 25.69388..."
1,029D39EC333C0D4CB504,ROI_1,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((98.23591 25.67962, 98.24014 25.70798..."
2,0AF83B5CE83C0D5A80F1,ROI_2,None,None,None,None,None,-1,0,-1,NaN,None,"POLYGON ((98.30643 25.66617, 98.29132 25.71977..."


In [ ]:
#Creating Geodataframe (steps: 1. Convert GeoDataFrame to Earth Engine Geometry / 2. Creates a buffer around points or uses geometry directly )
def gdf_to_ee_geometry(gdf):
    # Simple approach: creating a buffer around all features
    # Get bounds of all geometries
    bounds = gdf.geometry.total_bounds  # [minx, miny, maxx, maxy]
    
    # Create a bounding box as a Polygon
    coords = [
        [bounds[0], bounds[1]],  # bottom-left
        [bounds[2], bounds[1]],  # bottom-right
        [bounds[2], bounds[3]],  # top-right
        [bounds[0], bounds[3]],  # top-left
        [bounds[0], bounds[1]]   # close the ring
    ]
    
    ee_geom = ee.Geometry.Polygon([coords])
    return ee_geom

# Convert to EE geometry
study_area = gdf_to_ee_geometry(gdf)
print("Study area converted to Earth Engine geometry (bounding box)")
print(f"Study area bounds: {study_area.bounds().getInfo()}")

Study area converted to Earth Engine geometry (bounding box)
Study area bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[98.22726678942718, 25.577707317607963], [98.45338589426612, 25.577707317607963], [98.45338589426612, 25.737495237414], [98.22726678942718, 25.737495237414], [98.22726678942718, 25.577707317607963]]]}


# Data Pre-processing Pipeline
- Cloud Masking          (included)
- Sunglint Correction    (excluded! - no need)
- Atmospheric Correction (excluded! - no need)

In [91]:
# --- Cloud masking using QA60 band ---
# Bit 10 = Cloud, Bit 11 = Cirrus

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloudBitMask  = 1 << 10
    cirrusBitMask = 1 << 11
    mask = (
        qa.bitwiseAnd(cloudBitMask).eq(0)
        .And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    )
    return image.updateMask(mask)

In [ ]:
# --- Full processing pipeline ---
def process_sentinel2(image):
    image = mask_s2_clouds(image)
    return image

# Satellite Imagery Extraction
- Set up the Timeframe (start date - end date)
- Set up Max cloud cover (20%)
- 


In [ ]:
# --- Filter and process Sentinel-2 collection ---
start_date     = '2025-01-01'
end_date       = '2025-12-31'
max_cloud_cover = 20 

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(study_area)                                          #study area
    .filterDate(start_date, end_date)                                  #start date - end date
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_cover))  #filter good images with less than 20% cloud
)

print(f"Found {s2_collection.size().getInfo()} Sentinel-2 images")

Found 19 Sentinel-2 images
Cloud-free median composite created for 2025-01-01 to 2025-12-31


In [ ]:
# Apply processing to every image, then create median composite
s2_processed = s2_collection.map(process_sentinel2)
s2_median    = s2_processed.median().clip(study_area)  # Clip to polygon
print(f"Cloud-free median composite created for {start_date} to {end_date}")

In [ ]:
# --- Export to Google Drive ---

# Bands: B2=Blue, B3=Green, B4=Red, B8=NIR, B11=SWIR
export_image = s2_median.select(['B2', 'B3', 'B4', 'B8', 'B11'])

task = ee.batch.Export.image.toDrive(                       
    image=export_image,
    description='Sentinel2_CloudFree_RareEarthSites',
    folder='EarthEngine_Exports',
    fileNamePrefix='Sentinel2_2025_Kachin',
    scale=10,
    region=study_area,
    fileFormat='GeoTIFF',
    crs='EPSG:4326',
    maxPixels=1e13
)

task.start()
print("Export task started. Check Tasks panel in GEE Code Editor or run task.status() to monitor.")
print(f"Task ID: {task.id}")

Export task started. Check Tasks panel in GEE Code Editor or run task.status() to monitor.
Task ID: ZH2JWESC25UDKD6AJM3YLBLW


In [ ]:
# --- Checking Export Status ---
import time

print("Monitoring export task...")

while True:
    status = task.status()
    state  = status['state']
    print(f"Status: {state}")
    if state in ['COMPLETED', 'FAILED', 'CANCELLED']:
        break
    time.sleep(30)  # Checking every 30 seconds

if state == 'COMPLETED':
    print("Export complete! Check your Google Drive > EarthEngine_Exports folder.")
else:
    print(f"Export ended with state: {state}")
    print(status)